# Dataset 3 - per-p embeddings (node2vec_v2_32)

Re-runs the **exact same** Node2Vec extraction used in notebook 08 (`node2vec_v2_32`, q=2) once for **each p**. Same ER network, same config - only the merged target changes at the end. Outputs one parquet per p next to the p0 baseline `node2vec_v2_32_dataset3_dataset.parquet`.

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.models.embeddings import Node2VecConfig, extract_embeddings

EMB_ROOT = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
DATASET_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3'
TARGETS_DIR = DATASET_DIR / 'targets'
N = 10000

def emb_path(name):
    return EMB_ROOT / 'dataset_3' / 'network_based' / name

e = pd.read_csv(DATASET_DIR / 'edges.csv')
edges = e.rename(columns={'from': 'Sourceid', 'to': 'Targetid', 'weight': 'Weights'})[['Sourceid', 'Targetid', 'Weights']]
nodes = pd.DataFrame({'index': np.arange(1, N + 1), 'feat': 1.0})   # dummy feat (Node2Vec ignores it)
TARGET_COL = 'log_systemic_risk_label'
P_LIST = ['p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

# Same config as the selected node2vec_v2_32 embedding (see notebooks 03/08).
cfg_node2vec_v2_32 = Node2VecConfig(embedding_dim=32, walk_length=20, context_size=10,
                                    walks_per_node=10, num_negative_samples=1, batch_size=256,
                                    lr=0.01, epochs=100, q=2, device='cpu')

In [2]:
for p in P_LIST:
    target_p = pd.read_csv(TARGETS_DIR / f'target_{p}.csv')
    emb, _ = extract_embeddings(edges, nodes, cfg_node2vec_v2_32)
    emb_cols = [c for c in emb.columns if c.startswith('emb_')]
    merged = (emb[['bank_id'] + emb_cols]
              .merge(target_p, on='bank_id', how='inner')[['bank_id'] + emb_cols + [TARGET_COL]])
    out = emb_path(f'node2vec_v2_32_dataset3_{p}_dataset.parquet')
    merged.to_parquet(out, index=False)
    print(f'{p:4s}  shape={merged.shape}  emb_cols={len(emb_cols)}  -> {out.name}')

p5    shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p5_dataset.parquet
p10   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p10_dataset.parquet
p15   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p15_dataset.parquet
p20   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p20_dataset.parquet
p25   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p25_dataset.parquet
p30   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p30_dataset.parquet
p35   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p35_dataset.parquet
p40   shape=(10000, 34)  emb_cols=32  -> node2vec_v2_32_dataset3_p40_dataset.parquet


In [3]:
files = sorted(emb_path('').glob('node2vec_v2_32_dataset3_p*_dataset.parquet'))
for f in files:
    print(f'{f.name:50s}  shape={pd.read_parquet(f).shape}')

node2vec_v2_32_dataset3_p10_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p15_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p20_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p25_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p30_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p35_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p40_dataset.parquet         shape=(10000, 34)
node2vec_v2_32_dataset3_p5_dataset.parquet          shape=(10000, 34)
